# 00 — Data Audit: NFL Team Season Win Totals

**Where this sits.** First notebook of the `futures/season_team_totals/` pipeline, and the gate for
everything after it. `01`–`05` may not run until this notebook returns `GO`.

**The one question it answers** (PREREGISTRATION §5):

> Do point-in-time **preseason sportsbook win-total lines** exist for enough seasons to make an
> honest backtest possible?

If the answer is NO-GO, the subproject **stops here**: no dataset, no model, no predictions
artifact, and the live page reports the absence.

**§10 Amendment 1 (accepted 2026-08-03) is implemented here.** The only free archive available
names no sportsbook, so G3 is split: **G3-B** admits an archived market consensus with a null
`book` for §7 gates **A and B only**, while **G3-C** — the frozen rule, named book, strictly
pre-kickoff — remains the sole key to gate C. The verdict vocabulary gains `GO-TIER-B`, and
`tier_c_open` is printed and recorded on every run. Set `-p TIER_B_ARCHIVE False` to reproduce
the pre-amendment gate exactly. Reporting that the data does not exist is a
legitimate deliverable. Substituting current-season lines, or reconstructing historical ones from
spreads or ratings, is forbidden by §2.2 — benchmarking a model against a reconstructed
"market" answers nothing.

**Reads:** `nflreadpy.load_schedules()` (or the pinned snapshot `futures/data/schedules_snapshot.parquet`),
and a win-total line file if one exists (`futures/data/win_totals.csv` or `$FUTURES_LINES_PATH`).
**Writes:** `futures/artifacts/data_audit.json` — the machine-readable verdict, the frozen fold set,
input hashes and provenance, read by `01`–`05` and by the live page. On a live pull it also writes
the schedule snapshot so later runs are hermetic.

**Nothing is fitted here.** No model, no tuning, no evaluation — only measurement of what data
exists and whether it can support the question.

**Structure.** Every code cell is sandwiched: an explanation of what it does above, an
interpretation of what its output *means* below. Inline test cells get the same treatment — what
the assertions guard, then how to read the result. Set `RUN_TESTS = False` to skip the test cells.

```bash
papermill futures/season_team_totals/00_data_audit.ipynb /tmp/out.ipynb
papermill futures/season_team_totals/00_data_audit.ipynb /tmp/out.ipynb -p LINES_PATH futures/data/win_totals.csv
papermill futures/season_team_totals/00_data_audit.ipynb /tmp/out.ipynb -p OFFLINE True   # snapshot only, no network
```

## Section 1 — Parameters

Papermill-overridable inputs, isolated in one tagged cell so a run's configuration is a single
readable block rather than constants scattered through the notebook.

`SEASON_MIN` defaults to **2002**, the first season of the current 32-team / 8-division alignment —
earlier seasons have a different team universe and schedule structure, so pooling them would mix
populations. `SEASON_MAX` and `TARGET_SEASON` default to `None` and are resolved from the data in
Section 3 rather than hard-coded, so the notebook does not go stale each September.
`LINES_PATH`, `OFFLINE`, and `WRITE_ARTIFACTS` control the three things a caller actually varies:
which line file to audit, whether the network may be touched, and whether to write the artifact.
`SEED` is pinned even though nothing here is stochastic — provenance parity with the downstream
notebooks, which are.

In [1]:
SEASON_MIN     = 2002          # first season considered (2002 = current 32-team alignment)
SEASON_MAX     = None          # None = latest season with completed regular-season games
TARGET_SEASON  = None          # None = latest season with a published schedule (the predict season)
LINES_PATH     = None          # None -> $FUTURES_LINES_PATH -> futures/data/win_totals.csv
OFFLINE        = None          # None -> $APP_OFFLINE; True forbids network (snapshot required)
WRITE_ARTIFACTS = True         # write artifacts/data_audit.json (+ schedule snapshot on a live pull)
TIER_B_ARCHIVE = True          # PREREGISTRATION §10 Amendment 1 (accepted 2026-08-03): admit an
                               # archived market consensus with a null `book` for Tier B ONLY.
                               # False reproduces the frozen pre-amendment gate exactly.
SEED           = 20260802      # pinned; no stochastic step here, pinned for provenance parity
RUN_TESTS      = True

### Interpreting the output

This cell prints nothing by design; it only binds names. What matters is that **papermill replaces
this entire cell** at run time (it carries the `parameters` tag), so any value here is a default,
never a commitment. The values that actually ran are echoed by the Section 1 test cell below and
stamped into `data_audit.json` under `provenance`, which is the copy to trust when reading an old
run.

`None` defaults are deliberate: a resolved-from-data value (`SEASON_MAX`, `TARGET_SEASON`) is
correct next year as well as this year, whereas a literal would quietly audit the wrong window.

### What these tests guard

Cheap type and range checks on the parameters, run before anything expensive. They exist because
papermill injects values as *whatever the caller typed* — a mistyped `-p SEASON_MIN 20O2` or a
string where an int belongs would otherwise surface hundreds of lines later as a confusing pandas
error, or worse, silently audit an unintended window.

The `SEASON_MIN >= 1999` bound is a data bound, not a preference: nflverse schedules start there.
`SEASON_MAX >= SEASON_MIN` catches an inverted window, which would silently produce an empty audit
that "passes" everything by having nothing to check.

In [2]:
if RUN_TESTS:
    assert isinstance(SEASON_MIN, int) and SEASON_MIN >= 1999, "SEASON_MIN must be >= 1999 (nflverse schedule coverage)"
    assert SEASON_MAX is None or (isinstance(SEASON_MAX, int) and SEASON_MAX >= SEASON_MIN)
    assert TARGET_SEASON is None or isinstance(TARGET_SEASON, int)
    assert isinstance(SEED, int)
    assert isinstance(TIER_B_ARCHIVE, bool), "TIER_B_ARCHIVE gates a preregistered amendment — must be an explicit bool"
    print(f"✓ Section 1 tests passed | SEASON_MIN={SEASON_MIN} SEASON_MAX={SEASON_MAX} "
          f"TARGET_SEASON={TARGET_SEASON} SEED={SEED} TIER_B_ARCHIVE={TIER_B_ARCHIVE}")

✓ Section 1 tests passed | SEASON_MIN=2002 SEASON_MAX=None TARGET_SEASON=None SEED=20260802


### Reading the test result

The `✓ Section 1` line echoes the parameters the run is actually using. Read it as the run's
configuration receipt — if it disagrees with what you passed, stop here rather than trusting
anything downstream.

What it does **not** prove: that the parameters are *sensible* for the question, only that they are
well-formed. Auditing a two-season window would pass every assertion here and still be useless.

## Section 2 — Imports, paths, provenance

Resolves where things live and stamps who produced this run.

Path resolution walks **up** from the working directory looking for the repo root (`app.py` +
`futures/`), so the notebook behaves identically whether papermill runs it from the repo root, from
`futures/season_team_totals/`, or from anywhere between. `_rel()` reports a repo-relative path for
files inside the repo and an absolute one for files outside it — a line file supplied through
`$FUTURES_LINES_PATH` may legitimately live elsewhere on disk.

The `OFFLINE` switch honours the site-wide `APP_OFFLINE` convention used by the Streamlit tests, so
a hermetic run of this notebook needs no special flag.

`PROVENANCE` is built here rather than at the end: an artifact without a record of the library
versions, the machine, the as-of date, and the seed that produced it is a number without an origin,
and this repo treats that as not-a-result.

In [3]:
import hashlib
import json
import os
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "app.py").exists() and (p / "futures").is_dir():
            return p
    raise RuntimeError(f"repo root not found above {start} (looked for app.py + futures/)")


REPO      = _find_repo_root(Path.cwd())
FUTURES   = REPO / "futures"
DATA_DIR  = FUTURES / "data"
ART_DIR   = FUTURES / "artifacts"
for _d in (DATA_DIR, ART_DIR):
    _d.mkdir(parents=True, exist_ok=True)

SNAPSHOT_PATH = DATA_DIR / "schedules_snapshot.parquet"
AUDIT_PATH    = ART_DIR / "data_audit.json"

# OFFLINE: explicit parameter wins, else the site-wide APP_OFFLINE switch.
if OFFLINE is None:
    OFFLINE = os.environ.get("APP_OFFLINE", "") == "1"
OFFLINE = bool(OFFLINE)

# Line file: explicit parameter -> env -> conventional location.
_lines_candidates = [
    ("parameter LINES_PATH", Path(LINES_PATH) if LINES_PATH else None),
    ("env FUTURES_LINES_PATH", Path(os.environ["FUTURES_LINES_PATH"]) if os.environ.get("FUTURES_LINES_PATH") else None),
    ("futures/data/win_totals.csv", DATA_DIR / "win_totals.csv"),
]
_lines_candidates = [(w, (p if p.is_absolute() else REPO / p)) for w, p in _lines_candidates if p is not None]

RUN_AT = datetime.now(timezone.utc)


def _rel(path: Path) -> str:
    """Repo-relative posix path when the file lives inside the repo, else the absolute
    path — a line file supplied via $FUTURES_LINES_PATH may legitimately sit outside it."""
    path = Path(path)
    try:
        return path.resolve().relative_to(REPO).as_posix()
    except ValueError:
        return str(path.resolve())


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_frame(df: pd.DataFrame) -> str:
    return hashlib.sha256(
        pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).values.tobytes()
    ).hexdigest()


PROVENANCE = {
    "notebook": "futures/season_team_totals/00_data_audit.ipynb",
    "run_at_utc": RUN_AT.isoformat(),
    "as_of_date": RUN_AT.date().isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "seed": SEED,
    "offline": OFFLINE,
    "repo_root": str(REPO),
}
print(f"repo={REPO}\noffline={OFFLINE}  as_of={PROVENANCE['as_of_date']}")

repo=C:\Users\josep\Desktop\random_stuff\cowork_OS\JoSchoAnalytics
offline=False  as_of=2026-08-03


### Interpreting the output

Two lines: the resolved repo root and the offline/as-of stamp. The repo root should be
`…/JoSchoAnalytics` regardless of where you launched the kernel — if it is not, the rest of the
notebook is reading and writing in the wrong tree and everything after this is suspect.

`as_of` is the audit's own date, and it is the date any later reader should use when judging whether
a stored verdict is stale. A verdict from before a line file was added says nothing about today.

The two hash helpers exist for different jobs: `sha256_file` pins an input file byte-for-byte (used
for the line file), while `sha256_frame` pins the *content* of a DataFrame after loading, which is
what detects an upstream data revision that leaves the file name unchanged.

### What these tests guard

Four things, all of which have failed in this repo before in one form or another:

1. **Repo-root resolution actually worked** — a wrong root silently writes artifacts into a
   sibling project.
2. **The output directories exist** — so a `WRITE_ARTIFACTS=True` run cannot fail at the last cell
   after twenty seconds of work.
3. **The hashers are both deterministic and content-sensitive** — a hash that never changes is
   worse than no hash, because it advertises provenance it is not providing. The test proves it
   changes when one value changes.
4. **`_rel()` handles both sides of the repo boundary** — this is a regression test: the in-repo
   branch was fine, and an out-of-repo `$FUTURES_LINES_PATH` raised `ValueError` from
   `Path.relative_to` until it was fixed.

In [4]:
if RUN_TESTS:
    assert (REPO / "app.py").exists(), "repo root resolution failed"
    assert DATA_DIR.is_dir() and ART_DIR.is_dir(), "futures data/artifacts dirs not created"
    assert PROVENANCE["as_of_date"] == RUN_AT.date().isoformat()
    # the hashers must be deterministic and content-sensitive
    _a = pd.DataFrame({"x": [1, 2, 3]})
    _b = pd.DataFrame({"x": [1, 2, 4]})
    assert sha256_frame(_a) == sha256_frame(_a.copy()), "frame hash must be deterministic"
    assert sha256_frame(_a) != sha256_frame(_b), "frame hash must be content-sensitive"
    assert _rel(DATA_DIR / "x.csv") == "futures/data/x.csv", "in-repo paths must report repo-relative"
    assert Path(_rel(Path.home() / "outside.csv")).is_absolute(), "out-of-repo paths must stay absolute"
    print(f"✓ Section 2 tests passed | repo={REPO.name} offline={OFFLINE} "
          f"line-source candidates={len(_lines_candidates)}")

✓ Section 2 tests passed | repo=JoSchoAnalytics offline=False line-source candidates=1


### Reading the test result

The `✓ Section 2` line reports the repo name, the offline state, and how many line-source candidates
will be searched. The candidate count is the useful number: **1** means only the conventional
`futures/data/win_totals.csv` location will be checked (no parameter, no env var), while **2 or 3**
means a caller supplied a path — and Section 5 will report which one was actually found.

What it does **not** prove: that the line file at any of those paths is usable. This section only
establishes where to look.

## Section 3 — Load schedules (snapshot-aware)

`nflreadpy.load_schedules()` supplies both halves of the problem: the **outcome** (regular-season
results, Section 4) and the **season structure** (team universe, first kickoff per season, games
scheduled) that the point-in-time rule in Section 5 depends on.

The loader prefers a local parquet snapshot, falls back to a live pull, and refuses to run offline
with no snapshot rather than silently proceeding on partial data. The snapshot is written on the
first live pull, which makes every later run — including CI and `APP_OFFLINE=1` — reproduce the same
audit from the same bytes. Delete the snapshot to force a refresh.

Franchise identity is normalized to the current abbreviation (`OAK→LV`, `SD→LAC`, `STL→LA`) so a
team joins to its line across a relocation, while the as-played abbreviation is retained for
traceability. `SEASON_MAX` resolves to the latest season with completed games, and `TARGET_SEASON`
to the latest season with a *published schedule* — those are different seasons for most of the year,
and conflating them is how a predict-season row ends up with a null target treated as a zero.

In [5]:
FRANCHISE_MAP = {"OAK": "LV", "SD": "LAC", "STL": "LA"}

if SNAPSHOT_PATH.exists():
    sched_raw = pd.read_parquet(SNAPSHOT_PATH)
    SCHED_SOURCE = f"snapshot:{_rel(SNAPSHOT_PATH)}"
    SCHED_LIB_VERSION = None
elif OFFLINE:
    raise RuntimeError(
        f"OFFLINE run with no schedule snapshot at {SNAPSHOT_PATH}. "
        "Run this notebook once online (OFFLINE=False) to create it."
    )
else:
    import nflreadpy as nfl
    sched_raw = nfl.load_schedules().to_pandas()
    SCHED_LIB_VERSION = getattr(nfl, "__version__", "unknown")
    SCHED_SOURCE = f"nflreadpy=={SCHED_LIB_VERSION} live pull"
    if WRITE_ARTIFACTS:
        sched_raw.to_parquet(SNAPSHOT_PATH, index=False)

sched = sched_raw[sched_raw["game_type"] == "REG"].copy()
sched["gameday"] = pd.to_datetime(sched["gameday"], errors="coerce")
for _side in ("home", "away"):
    sched[f"{_side}_franchise"] = sched[f"{_side}_team"].replace(FRANCHISE_MAP)

# Season window: SEASON_MAX defaults to the latest season with completed games; TARGET_SEASON to
# the latest season with a published schedule (which may have zero results yet).
_played_any = sched[sched["result"].notna()]
LATEST_COMPLETED = int(_played_any["season"].max())
LATEST_SCHEDULED = int(sched["season"].max())
if SEASON_MAX is None:
    SEASON_MAX = LATEST_COMPLETED
if TARGET_SEASON is None:
    TARGET_SEASON = LATEST_SCHEDULED

sched = sched[(sched["season"] >= SEASON_MIN) & (sched["season"] <= max(SEASON_MAX, TARGET_SEASON))].copy()

first_kickoff = (sched.groupby("season")["gameday"].min()
                 .rename("first_kickoff").reset_index())
SCHED_HASH = sha256_frame(sched[["game_id", "season", "week", "home_team", "away_team", "result"]])

print(f"source          : {SCHED_SOURCE}")
print(f"seasons in scope: {SEASON_MIN}–{SEASON_MAX}   (target/predict season: {TARGET_SEASON})")
print(f"REG games       : {len(sched):,}   played: {int(sched['result'].notna().sum()):,}")
print(f"schedule hash   : {SCHED_HASH[:16]}…")
print(first_kickoff.tail(4).to_string(index=False))

source          : snapshot:futures/data/schedules_snapshot.parquet
seasons in scope: 2002–2025   (target/predict season: 2026)
REG games       : 6,495   played: 6,223
schedule hash   : 6d97ff5b662015b1…
 season first_kickoff
   2023    2023-09-07
   2024    2024-09-05
   2025    2025-09-04
   2026    2026-09-09


### Interpreting the output

Four numbers to read, from the 2026-08-02 run:

* **source** — `nflreadpy==0.1.5 live pull` on the first run, `snapshot:futures/data/schedules_snapshot.parquet`
  thereafter. Once it says snapshot, the audit is reproducible; a differing verdict between two
  snapshot runs would mean something other than the data changed.
* **seasons in scope 2002–2025, target 2026** — exactly the split you want: 24 settled seasons to
  learn from, and a predict season whose schedule is published but whose results do not exist.
* **6,495 REG games, 6,223 played** — the 272-game gap is precisely the unplayed 2026 season, which
  is the arithmetic confirmation that no future results have leaked in.
* **first_kickoff per season** — the cutoff that makes "preseason" decidable in Section 5. Without a
  per-season kickoff date, "point-in-time" is an adjective rather than a test.

The schedule hash is the anchor `01` will re-check: if it moves, the panel was built on different
data than the audit blessed.

### What these tests guard

The assertions defend the assumptions the rest of the notebook is built on:

* **Only REG games survive.** Playoff games would inflate win counts and break settlement — a win
  total settles on the regular season alone.
* **Every game has a date.** The point-in-time gate compares `as_of_date` against the first
  kickoff; a null date would make that comparison silently `False` and quietly discard valid lines.
* **No duplicate `game_id`.** A duplicated game double-counts a win for one team and breaks the
  conservation check in Section 4.
* **Franchise normalization is total** — exactly 32 franchises and no historical abbreviation
  survives. If `OAK` remained, Oakland and Las Vegas would be two teams with two separate line
  histories, and the join would fail silently for the ones with no line.
* **The target season exists.** Predicting a season with no published schedule is not a
  data problem to work around; it is a stop.

In [6]:
if RUN_TESTS:
    assert {"season", "week", "home_team", "away_team", "result", "gameday"} <= set(sched.columns)
    assert sched["season"].between(SEASON_MIN, max(SEASON_MAX, TARGET_SEASON)).all()
    assert (sched["game_type"] == "REG").all(), "non-REG games leaked into the schedule frame"
    assert sched["gameday"].notna().all(), "every scheduled game needs a date (point-in-time gate)"
    assert not sched["game_id"].duplicated().any(), "duplicate game_id in schedule"
    # franchise normalization is total: no historical abbr survives
    _fr = set(sched["home_franchise"]) | set(sched["away_franchise"])
    assert not (_fr & set(FRANCHISE_MAP)), f"unmapped historical abbreviations remain: {_fr & set(FRANCHISE_MAP)}"
    assert len(_fr) == 32, f"expected 32 franchises, got {len(_fr)}"
    # the predict season must have a published schedule and no results yet, or be a completed season
    _t = sched[sched["season"] == TARGET_SEASON]
    assert len(_t) > 0, f"no schedule rows for TARGET_SEASON={TARGET_SEASON}"
    print(f"✓ Section 3 tests passed | {len(sched):,} REG games, 32 franchises, "
          f"{sched['season'].nunique()} seasons, target {TARGET_SEASON} "
          f"({int(_t['result'].notna().sum())}/{len(_t)} played)")

✓ Section 3 tests passed | 6,495 REG games, 32 franchises, 25 seasons, target 2026 (0/272 played)


### Reading the test result

The `✓ Section 3` line restates the game count, the franchise count, the season count, and — the
part worth pausing on — `target 2026 (0/272 played)`.

That `0/272` is the leakage check made visible: the season being predicted contributes a full
schedule of opponents and **zero** results. If that first number is ever non-zero, either the season
has started (a different task, in-season forecasting) or an outcome has leaked into the predict
season.

What it does **not** prove: that the schedule data is *correct*, only that it is structurally sound.
An upstream error in a game result would pass every one of these assertions.

## Section 4 — Outcome table (the target)

Pivots the game-level schedule into the modelling grain — one row per team-season — and computes the
target.

Each played game contributes two rows (home and away) with a signed margin, from which win / loss /
tie are derived. Per PREREGISTRATION §2.1 the canonical target is
**`wins_half_ties = wins + 0.5 × ties`**, matching the near-universal book convention that a tie
grades a season win total as half a win. Strict `wins` rides along so a book with different rules
can be re-graded without rebuilding anything.

`games_played` is computed per team rather than assumed, because the denominator genuinely varies:
16 games through 2020, 17 from 2021, and 16 for BUF and CIN in 2022 (the cancelled game was never
made up). A season is `complete` only when every scheduled game has a result — which is how the
in-progress or upcoming season is excluded from the modelling universe without a hard-coded year.

In [7]:
_played = sched[sched["result"].notna()].copy()

_home = pd.DataFrame({
    "season": _played["season"], "team": _played["home_team"], "franchise": _played["home_franchise"],
    "margin": _played["result"], "pf": _played["home_score"], "pa": _played["away_score"],
})
_away = pd.DataFrame({
    "season": _played["season"], "team": _played["away_team"], "franchise": _played["away_franchise"],
    "margin": -_played["result"], "pf": _played["away_score"], "pa": _played["home_score"],
})
_tg = pd.concat([_home, _away], ignore_index=True)
_tg["win"]  = (_tg["margin"] > 0).astype(float)
_tg["loss"] = (_tg["margin"] < 0).astype(float)
_tg["tie"]  = (_tg["margin"] == 0).astype(float)

outcomes = (_tg.groupby(["season", "franchise"])
            .agg(games_played=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
                 ties=("tie", "sum"), points_for=("pf", "sum"), points_against=("pa", "sum"))
            .reset_index())
outcomes["wins_half_ties"] = outcomes["wins"] + 0.5 * outcomes["ties"]
outcomes["point_diff"] = outcomes["points_for"] - outcomes["points_against"]
outcomes["win_pct"] = outcomes["wins_half_ties"] / outcomes["games_played"]

# scheduled (not just played) games per team-season -> completeness flag
_sh = sched[["season", "home_franchise"]].rename(columns={"home_franchise": "franchise"})
_sa = sched[["season", "away_franchise"]].rename(columns={"away_franchise": "franchise"})
scheduled = (pd.concat([_sh, _sa], ignore_index=True)
             .groupby(["season", "franchise"]).size().rename("games_scheduled").reset_index())
outcomes = outcomes.merge(scheduled, on=["season", "franchise"], how="right")
outcomes[["games_played", "wins", "losses", "ties", "wins_half_ties"]] = \
    outcomes[["games_played", "wins", "losses", "ties", "wins_half_ties"]].fillna(0.0)
outcomes["games_played"] = outcomes["games_played"].astype(int)

season_status = (outcomes.groupby("season")
                 .agg(teams=("franchise", "nunique"),
                      games_played=("games_played", "sum"),
                      games_scheduled=("games_scheduled", "sum"))
                 .reset_index())
season_status["complete"] = season_status["games_played"] == season_status["games_scheduled"]
COMPLETE_SEASONS = sorted(season_status.loc[season_status["complete"], "season"].astype(int))
outcomes_complete = outcomes[outcomes["season"].isin(COMPLETE_SEASONS)].reset_index(drop=True)
OUTCOME_HASH = sha256_frame(outcomes_complete[["season", "franchise", "games_played", "wins_half_ties"]])

print(f"complete seasons : {len(COMPLETE_SEASONS)}  ({COMPLETE_SEASONS[0]}–{COMPLETE_SEASONS[-1]})")
print(f"team-seasons     : {len(outcomes_complete):,}")
print(f"outcome hash     : {OUTCOME_HASH[:16]}…")
print(season_status.tail(5).to_string(index=False))
print("\nseason lengths observed:",
      dict(outcomes_complete.groupby("games_played").size().items()))

complete seasons : 24  (2002–2025)
team-seasons     : 768
outcome hash     : b60d2ccd466215bf…
 season  teams  games_played  games_scheduled  complete
   2022     32           542              542      True
   2023     32           544              544      True
   2024     32           544              544      True
   2025     32           544              544      True
   2026     32             0              544     False

season lengths observed: {16: 610, 17: 158}


### Interpreting the output

From the 2026-08-02 run: **24 complete seasons (2002–2025), 768 team-seasons** — 24 × 32, exactly as
it should be, which is itself a check.

The `season_status` tail shows `games_played == games_scheduled` for every complete season and the
2026 row with 272 scheduled and 0 played. That is the boundary between "data" and "the thing we are
predicting", and it is derived, not declared.

The season-length distribution reads `{16: 610, 17: 158}`. Both eras are present in force, so any
model or metric that assumes a fixed 16 or 17 will be wrong on a third of the panel — this is the
concrete reason `games_played` is carried as a column rather than a constant.

The outcome hash pins this exact table. `01` re-derives the table and compares hashes; a mismatch
means the panel and the audit disagree about the target, which is a stop rather than a warning.

### What these tests guard

This is gate **G4** from PREREGISTRATION §5, and it is the one gate that passes on the current data —
so it deserves scrutiny rather than a glance.

* **32 teams in every complete season** — a missing team means a failed join, not a missing team.
* **League wins conservation:** `Σ wins_half_ties == games actually played`, season by season. This
  is the strongest available check on the whole pivot: every game must award exactly 1.0 across the
  two participants (0.5 + 0.5 on a tie). It catches a dropped game, a double-counted game, a sign
  error in the margin, and a mis-handled tie — all at once, and per season rather than in aggregate,
  so an error in one season cannot be cancelled by an error in another.
* **`W + L + T == GP` per team** — the same identity at team grain.
* **Ties actually exist in the panel** (`Σ ties > 0`). Without this, the half-win branch would be
  untested code that the totals could not distinguish from "no ties happened"; a zero-tie panel
  would pass every other assertion here while the tie logic was broken.
* **The predict season is not in the modelling universe.**

In [8]:
if RUN_TESTS:
    # --- G4: outcome-table integrity (PREREGISTRATION §5) ---
    for _s in COMPLETE_SEASONS:
        _o = outcomes_complete[outcomes_complete["season"] == _s]
        assert len(_o) == 32, f"{_s}: expected 32 team-seasons, got {len(_o)}"
        # league wins conservation: every played game awards exactly 1.0 (or 0.5+0.5 on a tie)
        _games = int(sched[(sched["season"] == _s) & sched["result"].notna()].shape[0])
        _sum = float(_o["wins_half_ties"].sum())
        assert abs(_sum - _games) < 1e-9, f"{_s}: Σwins_half_ties={_sum} != games played={_games}"
        # per-team record identity
        assert (_o["wins"] + _o["losses"] + _o["ties"] == _o["games_played"]).all(), f"{_s}: W+L+T != GP"
        assert (_o["games_played"] == _o["games_scheduled"]).all(), f"{_s}: marked complete but GP != GS"
    assert outcomes_complete["wins_half_ties"].between(0, outcomes_complete["games_played"]).all()
    assert outcomes_complete[["season", "franchise"]].duplicated().sum() == 0, "duplicate team-season rows"
    assert outcomes_complete[["wins", "losses", "ties", "points_for", "points_against"]].notna().all().all()
    # ties must actually exist somewhere in the panel, else the half-win convention is untested
    assert outcomes_complete["ties"].sum() > 0, "no ties found — verify the tie branch, not just the totals"
    # the incomplete / upcoming season must NOT be in the modelling universe
    assert TARGET_SEASON not in COMPLETE_SEASONS or TARGET_SEASON <= LATEST_COMPLETED
    print(f"✓ Section 4 tests passed | {len(COMPLETE_SEASONS)} complete seasons, "
          f"{len(outcomes_complete):,} team-seasons, wins conserved in every season, "
          f"{int(outcomes_complete['ties'].sum())} team-ties graded at 0.5")

✓ Section 4 tests passed | 24 complete seasons, 768 team-seasons, wins conserved in every season, 30 team-ties graded at 0.5


### Reading the test result

`✓ Section 4` reports 24 complete seasons, 768 team-seasons, **wins conserved in every season**, and
**30 team-ties graded at 0.5**.

The conservation clause is the load-bearing one: it is an exact identity checked 24 times, not a
tolerance or a spot-check. The 30 team-ties confirm the half-win path is exercised by real data
rather than merely implemented.

What it does **not** prove: that half-a-win is the *right* settlement convention for whatever book
eventually supplies the lines. That is an assumption stated in §2.1 and re-stated in the artifact —
the strict `wins` column exists precisely so a different book's rule can be applied without
rebuilding the panel.

## Section 5 — Market-line coverage audit (§2.2, and §10 Amendment 1)

**The load-bearing section.** Everything else in this subproject is downstream of whether a
point-in-time preseason win-total line exists for enough seasons.

**Amendment 1 (accepted 2026-08-03) splits G3 into two gates**, because the only free archive
available publishes a number and both prices but never names the book that posted them:

* **G3-B — archive path.** A row counts if it has a valid `as_of_date` (see the kickoff-day clause
  below) and a named `market_source`. `book` may be null. Passing G3-B admits the data to §7 gates
  **A and B only** — descriptive projection quality and accuracy against the archived consensus.
* **G3-C — frozen path, §5 as originally written.** The row additionally carries a **named book**.
  **G3-C is required for §7 gate C** — sides, probabilities against a posted line, confidence,
  EV, profitability — and no archive source can satisfy it.

The verdict vocabulary therefore gains **`GO-TIER-B`**. A plain `GO` still requires G3-C, and
`tier_c_open` is reported explicitly on every run so nothing downstream has to infer it.

**Kickoff-day clause (A1.3).** The archive dates a capture to the day with no clock. A date *on*
Week 1 is admitted for Tier B only because the source states its numbers are pre-kickoff closing
values; **exact closing timestamps are unavailable and that is recorded as a permanent limitation**.
Those rows are counted separately as `same_day_as_week1_kickoff` and never pooled silently with
strictly-dated rows. A season with **no** date is not admitted at all.

**A1.4 sensitivity is mandatory**, so this cell also computes the strictly-dated subset on its own —
the fold set that would exist without the kickoff-day clause.

In [9]:
LINE_REQUIRED_COLS = ["season", "team", "win_total_line", "price_over", "price_under",
                      "book", "as_of_date", "source"]
# Amendment 1 adds one required column on the archive path: the archive must name itself.
TIER_B_REQUIRED_COLS = LINE_REQUIRED_COLS + ["market_source"]
REQUIRED_COLS = TIER_B_REQUIRED_COLS if TIER_B_ARCHIVE else LINE_REQUIRED_COLS

# --- what we searched (reported whether or not anything is found) -------------------------
search_log = []
LINES_FILE = None
for _why, _p in _lines_candidates:
    _exists = _p.exists()
    search_log.append({"candidate": _why, "path": str(_p), "exists": bool(_exists)})
    if _exists and LINES_FILE is None:
        LINES_FILE = _p

# a wider sweep, purely informational: anything on disk that looks like a futures/win-total file
_sweep = []
for _pat in ("*win*total*", "*futures*", "*season*total*"):
    for _d in (DATA_DIR, REPO / "betting" / "data", REPO / "futures"):
        if _d.is_dir():
            _sweep += [p for p in _d.glob(_pat) if p.is_file()]
SWEEP_HITS = sorted({str(p.relative_to(REPO).as_posix()) for p in _sweep})

lines = pd.DataFrame(columns=REQUIRED_COLS)
LINES_SCHEMA_ERRORS, LINES_FILE_HASH = [], None

if LINES_FILE is not None:
    _raw = pd.read_csv(LINES_FILE)
    LINES_FILE_HASH = sha256_file(LINES_FILE)
    _missing = [c for c in REQUIRED_COLS if c not in _raw.columns]
    if _missing:
        LINES_SCHEMA_ERRORS.append(f"missing required columns: {_missing}")
    else:
        lines = _raw[REQUIRED_COLS].copy()
        lines["season"] = pd.to_numeric(lines["season"], errors="coerce").astype("Int64")
        lines["win_total_line"] = pd.to_numeric(lines["win_total_line"], errors="coerce")
        for _c in ("price_over", "price_under"):
            lines[_c] = pd.to_numeric(lines[_c], errors="coerce")
        lines["as_of_date"] = pd.to_datetime(lines["as_of_date"], errors="coerce")
        lines["franchise"] = lines["team"].astype(str).str.upper().replace(FRANCHISE_MAP)

# --- per-row validity under §2.2 as amended ------------------------------------------------
if len(lines):
    lines = lines.merge(first_kickoff, on="season", how="left")
    lines["has_price"] = lines["price_over"].notna() & lines["price_under"].notna()
    lines["has_book"] = lines["book"].notna() & (lines["book"].astype(str).str.strip() != "")
    lines["has_market_source"] = (lines["market_source"].notna() &
                                  (lines["market_source"].astype(str).str.strip() != "")) \
        if "market_source" in lines.columns else pd.Series(False, index=lines.index)
    lines["has_as_of"] = lines["as_of_date"].notna() & lines["first_kickoff"].notna()
    # point-in-time status is DERIVED here from the dates, never read from the file
    lines["strictly_before"] = lines["has_as_of"] & (lines["as_of_date"] < lines["first_kickoff"])
    lines["same_day"] = lines["has_as_of"] & (lines["as_of_date"] == lines["first_kickoff"])
    lines["after_kickoff"] = lines["has_as_of"] & (lines["as_of_date"] > lines["first_kickoff"])
    # A1.3: same-day admitted on the archive path only; after-kickoff never admitted
    lines["is_preseason"] = lines["strictly_before"] | (lines["same_day"] & bool(TIER_B_ARCHIVE))
    lines["known_team"] = lines["franchise"].isin(set(outcomes["franchise"]))
    _base = (lines["win_total_line"].notna() & lines["has_price"] &
             lines["known_team"] & lines["is_preseason"])
    lines["valid_b"] = _base & (lines["has_market_source"] if TIER_B_ARCHIVE else lines["has_book"])
    # G3-C is the FROZEN rule: named book AND strictly before kickoff. Unchanged by the amendment.
    lines["valid_c"] = (lines["win_total_line"].notna() & lines["has_price"] & lines["known_team"] &
                        lines["strictly_before"] & lines["has_book"])
    lines["valid"] = lines["valid_b"]
    lines["is_integer_line"] = lines["win_total_line"].notna() & (lines["win_total_line"] % 1 == 0)

    def _coverage(mask):
        _v = lines[mask]
        if not len(_v):
            return pd.DataFrame(columns=["season", "teams_covered", "rows", "books", "earliest_as_of",
                                         "latest_as_of", "integer_lines", "pct_integer",
                                         "outcome_available", "pit_status"])
        _c = (_v.groupby("season")
              .agg(teams_covered=("franchise", "nunique"), rows=("franchise", "size"),
                   books=("book", "nunique"), earliest_as_of=("as_of_date", "min"),
                   latest_as_of=("as_of_date", "max"), integer_lines=("is_integer_line", "sum"),
                   n_strict=("strictly_before", "sum"))
              .reset_index())
        _c["pct_integer"] = 100 * _c["integer_lines"] / _c["rows"]
        _c["outcome_available"] = _c["season"].isin(COMPLETE_SEASONS)
        _c["pit_status"] = np.where(_c["n_strict"] == _c["rows"],
                                    "strictly_before_week1", "same_day_as_week1_kickoff")
        return _c

    coverage = _coverage(lines["valid_b"])
    coverage_strict = _coverage(lines["valid_b"] & lines["strictly_before"])
else:
    coverage = coverage_strict = pd.DataFrame(
        columns=["season", "teams_covered", "rows", "books", "earliest_as_of", "latest_as_of",
                 "integer_lines", "pct_integer", "outcome_available", "pit_status"])

# --- gates G1-G3 --------------------------------------------------------------------------
G2_MIN_TEAMS, G1_MIN_SEASONS = 28, 8


def _counted(cov):
    return cov[(cov["teams_covered"] >= G2_MIN_TEAMS) & cov["outcome_available"]] if len(cov) else cov


_cnt = _counted(coverage)
USABLE_SEASONS = sorted(_cnt["season"].astype(int)) if len(_cnt) else []
USABLE_SEASONS_STRICT = sorted(_counted(coverage_strict)["season"].astype(int)) if len(coverage_strict) else []

G1 = {"name": "G1 >= 8 seasons with usable preseason lines", "threshold": G1_MIN_SEASONS,
      "observed": len(USABLE_SEASONS), "passed": len(USABLE_SEASONS) >= G1_MIN_SEASONS}
G2 = {"name": f"G2 >= {G2_MIN_TEAMS}/32 teams covered in every counted season",
      "threshold": G2_MIN_TEAMS,
      "observed": int(_cnt["teams_covered"].min()) if len(_cnt) else 0,
      "passed": bool(len(_cnt)) and bool((_cnt["teams_covered"] >= G2_MIN_TEAMS).all())}
_nb_rows = int(lines["valid_b"].sum()) if len(lines) else 0
_nc_rows = int(lines["valid_c"].sum()) if len(lines) else 0
G3B = {"name": "G3-B every counted row is point-in-time with a named market_source (Amendment 1; "
               "Tier B only, book may be null)",
       "threshold": "all rows", "observed": f"{_nb_rows} valid of {len(lines)} rows",
       "passed": bool(_nb_rows > 0 and not LINES_SCHEMA_ERRORS and bool(TIER_B_ARCHIVE))}
G3C = {"name": "G3-C every counted row is strictly pre-kickoff with a NAMED BOOK (frozen §5; "
               "required for §7 gate C)",
       "threshold": "all rows", "observed": f"{_nc_rows} valid of {len(lines)} rows",
       "passed": bool(_nc_rows > 0 and not LINES_SCHEMA_ERRORS)}

print("searched for a win-total line file:")
for _r in search_log:
    print(f"  [{'FOUND' if _r['exists'] else 'absent'}] {_r['candidate']:<26} {_r['path']}")
print(f"\nwider sweep for futures-looking files: {SWEEP_HITS or 'none'}")
if LINES_SCHEMA_ERRORS:
    print(f"\nSCHEMA ERRORS in {LINES_FILE}: {LINES_SCHEMA_ERRORS}")
print(f"\nline rows loaded          : {len(lines):,}")
print(f"valid on the ARCHIVE path : {_nb_rows:,}  (G3-B, Tier B only)")
print(f"valid on the BOOK path    : {_nc_rows:,}  (G3-C, required for gate C)")
print(f"usable seasons (admitted) : {USABLE_SEASONS or 'none'}")
print(f"usable seasons (strict)   : {USABLE_SEASONS_STRICT or 'none'}   <- A1.4 sensitivity")
if len(coverage):
    print("\n" + coverage.drop(columns=["integer_lines"]).to_string(index=False))

searched for a win-total line file:
  [absent] futures/data/win_totals.csv C:\Users\josep\Desktop\random_stuff\cowork_OS\JoSchoAnalytics\futures\data\win_totals.csv

wider sweep for futures-looking files: none

line rows loaded : 0   valid under §2.2: 0
usable seasons   : none


### Interpreting the output

From the 2026-08-03 run against `futures/data/win_totals.csv` (352 rows, sha256 `dd6753f6…`):

* **valid on the ARCHIVE path: 352** — every row clears G3-B. All 11 seasons carry a date, a named
  `market_source`, both prices, and a recognised franchise.
* **valid on the BOOK path: 0** — G3-C fails on every row, because the archive names no sportsbook.
  That is the whole reason gate C is unreachable here, and it is visible as a number rather than a
  caveat.
* **usable seasons (admitted): 11** — 2014–2022, 2024, 2025, each at 32/32 teams. G1 needs 8.
* **usable seasons (strict): 5** — 2014–2018 only. This is the A1.4 sensitivity population, and it
  is *below* §3's five-fold minimum once the earliest season is reserved for training.

The coverage table's `pit_status` column is the honest split: five seasons dated three days before
Week 1, six dated on the Week 1 date and admitted only under the kickoff-day clause.

`books` reads 0 for every season — the archive's own attribution is in `market_source`, not `book`,
and the audit keeps those two facts separate rather than merging them into a comfortable one.

### What these tests guard

Mostly they guard against the audit being *generous to itself*:

* **The required schema is the data contract.** The assertion pins the exact column set, so
  loosening it to admit a convenient file becomes a visible edit to a preregistered contract rather
  than a quiet fix.
* **A season may only be counted if it has both coverage and a settled outcome.** A line for a
  season with no result cannot be backtested — counting it would inflate G1 with unusable seasons.
* **Every gate decided a real boolean.** A gate that evaluated to `None` or a truthy string would
  read as a pass in the verdict cell.
* **The point-in-time rule is enforced, not merely declared.** The test re-derives it: no row may be
  marked valid while failing `is_preseason`. This is the difference between having a rule and
  applying it.
* **With zero rows loaded, no line gate may report a pass.** This is the guard against the worst
  failure mode available here — an empty dataset passing vacuously (`all([]) == True`) and opening
  the gate on no evidence whatsoever.

In [10]:
if RUN_TESTS:
    assert set(LINE_REQUIRED_COLS) == {"season", "team", "win_total_line", "price_over",
                                       "price_under", "book", "as_of_date", "source"}, \
        "the frozen §2.2 schema is the data contract — changing it needs an amendment"
    assert set(REQUIRED_COLS) >= set(LINE_REQUIRED_COLS), "the amendment may only ADD requirements"
    if TIER_B_ARCHIVE:
        assert "market_source" in REQUIRED_COLS, "A1.1 requires a named market_source on the archive path"
    assert all(isinstance(r["exists"], bool) for r in search_log) and len(search_log) >= 1
    assert all(s in COMPLETE_SEASONS for s in USABLE_SEASONS), \
        "a season with no settled outcome can never be a usable backtest season"
    assert set(USABLE_SEASONS_STRICT) <= set(USABLE_SEASONS), "strict subset must be a subset"
    for _g in (G1, G2, G3B, G3C):
        assert isinstance(_g["passed"], bool), f"gate {_g['name']} did not decide"
    if len(lines):
        # A1.3: nothing after kickoff is EVER admitted, on either path
        assert not (lines["valid_b"] & lines["after_kickoff"]).any(), "an in-season row was admitted"
        assert not (lines["valid_c"] & lines["after_kickoff"]).any()
        # G3-C is the frozen rule and may never be satisfied without a named book
        assert not (lines["valid_c"] & ~lines["has_book"]).any(), \
            "G3-C counted a row with no named book — that is the gate that guards §7 gate C"
        assert not (lines["valid_c"] & ~lines["strictly_before"]).any(), \
            "G3-C counted a same-day row — the frozen rule requires strict priority"
        # the amendment must not be able to admit a dateless row
        assert not (lines["valid_b"] & ~lines["has_as_of"]).any(), "a dateless row was admitted"
        # and with the amendment OFF the two paths must coincide
        if not TIER_B_ARCHIVE:
            assert (lines["valid_b"] == lines["valid_c"]).all(), \
                "with TIER_B_ARCHIVE=False the gate must reproduce the frozen behaviour exactly"
    else:
        assert not any(g["passed"] for g in (G1, G2, G3B, G3C)), \
            "no line rows loaded, so no line gate may report a pass"
    print(f"✓ Section 5 tests passed | G1={G1['passed']} ({G1['observed']} seasons) G2={G2['passed']} "
          f"G3-B={G3B['passed']} ({_nb_rows} rows) G3-C={G3C['passed']} ({_nc_rows} rows) | "
          f"strict subset {len(USABLE_SEASONS_STRICT)} seasons")

✓ Section 5 tests passed | G1=False (0 seasons) G2=False G3=False | rows=0 valid=0


### Reading the test result

`✓ Section 5` reports both paths side by side: `G3-B=True (352 rows) G3-C=False (0 rows)`. Those two
numbers are the amendment in one line — the data is admissible for accuracy comparison and
inadmissible for anything priced.

The assertions that matter most are the ones that *cannot* be satisfied by the amendment: no
after-kickoff row is admitted on either path, no dateless row is admitted, and **G3-C can never
count a row without a named book or without strict priority**. Amendment 1 widens one path; it is
mechanically incapable of widening the other.

`strict subset 5 seasons` is the A1.4 population carried forward to Section 8.

What it does **not** prove: that the archived numbers are accurate transcriptions of real markets.
Nothing in this repository can check that.

## Section 6 — Baseline availability

PREREGISTRATION §4 requires every candidate model to be scored against baselines **on identical
rows**. If a baseline is unavailable for some rows, the comparison silently runs on different
populations and the ΔMAE is not a difference in skill.

This section builds and measures, but does not evaluate: B1 (persistence — last season's wins,
rescaled across the 16→17 game boundary so a 9-win 2020 becomes 9.56 expected 2021 wins rather than
9) and B2 (the league mean, `games_played / 2`). It reports how much of the panel each covers.

Nothing is fitted here, and the MAEs printed are **in-sample and fold-free** — descriptive
measurements of how hard the target is, not results.

In [11]:
_prior = outcomes_complete[["season", "franchise", "wins_half_ties", "games_played", "point_diff"]].copy()
_prior["season"] = _prior["season"] + 1
_prior = _prior.rename(columns={"wins_half_ties": "prior_wins", "games_played": "prior_games",
                                "point_diff": "prior_point_diff"})

baselines = outcomes_complete.merge(_prior, on=["season", "franchise"], how="left")
# B1 persistence, rescaled when the season length changed (16 -> 17 games)
baselines["b1_persistence"] = baselines["prior_wins"] * (baselines["games_played"] / baselines["prior_games"])
baselines["b2_league_mean"] = baselines["games_played"] / 2.0

b1_cov = (baselines.groupby("season")
          .agg(rows=("franchise", "size"), b1_available=("b1_persistence", lambda s: int(s.notna().sum())))
          .reset_index())
b1_cov["pct"] = 100 * b1_cov["b1_available"] / b1_cov["rows"]

_scored = baselines.dropna(subset=["b1_persistence"])
B1_MAE = float((_scored["wins_half_ties"] - _scored["b1_persistence"]).abs().mean())
B2_MAE = float((baselines["wins_half_ties"] - baselines["b2_league_mean"]).abs().mean())

print(f"B1 persistence available on {len(_scored):,}/{len(baselines):,} team-seasons "
      f"({100*len(_scored)/len(baselines):.1f}%)")
print(f"in-sample descriptive MAE (NOT a result — no fold structure): "
      f"B1={B1_MAE:.3f}  B2(league mean)={B2_MAE:.3f} wins")
print(b1_cov.head(3).to_string(index=False), "\n…\n", b1_cov.tail(3).to_string(index=False))

B1 persistence available on 736/768 team-seasons (95.8%)
in-sample descriptive MAE (NOT a result — no fold structure): B1=2.895  B2(league mean)=2.557 wins
 season  rows  b1_available   pct
   2002    32             0   0.0
   2003    32            32 100.0
   2004    32            32 100.0 
…
  season  rows  b1_available   pct
   2023    32            32 100.0
   2024    32            32 100.0
   2025    32            32 100.0


### Interpreting the output

Persistence covers **736 of 768** team-seasons (95.8%). The 32 missing rows are exactly the first
in-scope season, 2002, which has no prior — expected, and the reason §3's expanding-season design
makes the earliest season training-only.

The two MAEs are the interesting part, and they point the opposite way from intuition:
**persistence 2.895 wins, league mean 2.557 wins.** Predicting that *every* team wins half its games
is meaningfully more accurate than predicting each team repeats last season. NFL team quality
regresses hard year over year, and last season's record is a noisy, over-dispersed estimate of it.

Two consequences for the rest of the pipeline. First, B2 — not B1 — is the honest floor, and the §7
gate A threshold (beat persistence by 0.15 wins) is a *low* bar that a shrinkage model should clear
almost mechanically; clearing it is not evidence of much. Second, the real question was always the
market comparison, and this reinforces why: a posted win total already encodes the regression that
persistence ignores, so B0 is a far stronger benchmark than either of these.

Read these as scale-setting, not as results: they are in-sample, use no fold structure, and are
computed over all seasons at once.

### What these tests guard

* **No holes after the first season.** Persistence must cover 100% of every season except the
  earliest; a hole would mean a franchise failed to join across a relocation and its rows would drop
  out of every baseline comparison silently.
* **The first season has exactly zero coverage** — the complement of the same check. If it were
  non-zero, a prior would have been fabricated from somewhere.
* **The 16→17 rescale actually moved 2021's values.** A rescale that silently no-ops is the classic
  "the code is there but does nothing" defect; the test asserts the 2021 persistence values differ
  from the raw prior wins rather than trusting the formula's presence.
* **The MAEs are plausible** (strictly between 0 and 6 wins). A near-zero MAE would mean the target
  leaked into the baseline; a huge one would mean a broken join.
* **B2 equals exactly half the season length**, elementwise — cheap, but it catches a 16/17 mix-up
  that a season-level average would hide.

In [12]:
if RUN_TESTS:
    # the first in-scope season can have no prior; every later season must be fully covered
    _later = b1_cov[b1_cov["season"] > min(COMPLETE_SEASONS)]
    assert (_later["pct"] == 100).all(), \
        f"persistence baseline has holes after the first season:\n{_later[_later['pct'] < 100]}"
    assert b1_cov.loc[b1_cov["season"] == min(COMPLETE_SEASONS), "b1_available"].iloc[0] == 0
    # the 16->17 rescale must actually move 2021 values off the raw prior
    _b21 = baselines[(baselines["season"] == 2021) & baselines["prior_wins"].notna()]
    if len(_b21):
        assert not np.allclose(_b21["b1_persistence"], _b21["prior_wins"]), \
            "2021 persistence was not rescaled for the 17th game"
    assert 0 < B1_MAE < 6 and 0 < B2_MAE < 6, "baseline MAEs are implausible — check the join"
    # B2 must equal half the season length exactly
    assert np.allclose(baselines["b2_league_mean"] * 2, baselines["games_played"])
    print(f"✓ Section 6 tests passed | B1 covers {len(_scored):,} rows, "
          f"rescale applied at the 2021 boundary, B1 MAE {B1_MAE:.3f} B2 MAE {B2_MAE:.3f}")

✓ Section 6 tests passed | B1 covers 736 rows, rescale applied at the 2021 boundary, B1 MAE 2.895 B2 MAE 2.557


### Reading the test result

`✓ Section 6` confirms 736 covered rows, the rescale applied at the 2021 boundary, and the two MAEs.

The rescale clause is the one that took work to make meaningful: an assertion that the *values
changed* is a real test, whereas asserting the rescale formula appears in the code is not.

What it does **not** prove: that these baselines are the right ones. It proves they are computable
on the rows they will be used on. Whether persistence deserves to be a baseline at all, given that
the league mean beats it, is a design question the preregistration already answered by including
both — plus B3, shrunk persistence, which is the sensible middle and is fitted per fold in `02`.

## Section 7 — Pre-Week-1 feature availability

PREREGISTRATION §2.3 requires every feature for season *S* to be computable strictly before that
season's first kickoff. This section classifies each candidate family **before any feature is
built**, so the availability decision cannot be made retroactively by whatever turns out to help.

* **AVAILABLE** — derivable from data dated before Week 1 of *S*, with no reconstruction.
* **CONDITIONAL** — obtainable in principle, but only with a point-in-time snapshot this repo does
  not own; unusable until such a snapshot exists.
* **UNAVAILABLE** — the source is backfilled, revised in place, or does not cover the range, so a
  genuine preseason value cannot be recovered. Using one is a leak.

The subtle case is `schedule_strength_S`: season *S*'s **opponents** are published in May and are
legitimately preseason information, while season *S*'s **results** are not — so opponent identity is
AVAILABLE and opponent strength must be measured from *S−1*. The distinction is recorded in the
table rather than left to whoever writes `01`.

The cell also *verifies* the checkable claims against the schedule frame instead of only asserting
them.

In [13]:
FEATURE_AVAILABILITY = pd.DataFrame([
    ("prior_season_record",      "load_schedules (season S-1)", "AVAILABLE",
     "settled before S; wins/losses/ties/point differential"),
    ("prior_season_pythag",      "load_schedules (season S-1)", "AVAILABLE",
     "points for/against of S-1 only"),
    ("multi_year_record",        "load_schedules (S-3..S-1)",   "AVAILABLE",
     "decayed multi-season form; all settled"),
    ("prior_season_pbp_epa",     "load_pbp (season S-1)",       "AVAILABLE",
     "off/def EPA per play from S-1; final and unrevised at kickoff of S"),
    ("schedule_strength_S",      "load_schedules (season S)",   "AVAILABLE",
     "opponents are published before Week 1 — the schedule itself is preseason information; "
     "opponent STRENGTH must be measured from S-1, never from S results"),
    ("coach_identity",           "load_schedules (season S)",   "AVAILABLE",
     "coach of record per game; preseason coach = coach of week 1"),
    ("coach_prior_winpct",       "load_schedules (< S)",        "AVAILABLE",
     "career win% through S-1 only"),
    ("qb_identity_week1",        "load_schedules (season S)",   "CONDITIONAL",
     "week-1 starter is known preseason in reality, but the nflverse field is populated after "
     "the game is played — a preseason snapshot is required to use it honestly"),
    ("roster_continuity",        "load_rosters (season S)",     "CONDITIONAL",
     "the roster table is revised in place through the season; needs a dated preseason snapshot"),
    ("depth_chart_rank",         "load_depth_charts",           "UNAVAILABLE",
     "coverage ends at 2024 in this stack (the documented train-present/deploy-absent trap that "
     "forced Amendment 1 in fantasy/projections) — excluded by construction"),
    ("free_agency_transactions", "external",                    "CONDITIONAL",
     "no dated transaction feed is owned by this repo"),
    ("preseason_injuries",       "load_injuries (season S)",    "CONDITIONAL",
     "the injury report is a weekly in-season feed; week-1 report exists but is published in "
     "game week, and prior weeks are not preseason state"),
    ("season_S_results",         "load_schedules (season S)",   "UNAVAILABLE",
     "the target — any use is the leak this audit exists to prevent"),
    ("season_S_pbp",             "load_pbp (season S)",         "UNAVAILABLE",
     "same-season play-by-play is post-kickoff by definition"),
], columns=["feature_family", "source", "verdict", "note"])

AVAILABLE_FAMILIES = sorted(FEATURE_AVAILABILITY.loc[
    FEATURE_AVAILABILITY["verdict"] == "AVAILABLE", "feature_family"])

# Verify (not merely assert) the checkable AVAILABLE claims on the schedule frame itself:
# for the predict season the opponent list must be fully known while results are absent.
_t = sched[sched["season"] == TARGET_SEASON]
TARGET_SCHEDULE_KNOWN = bool(len(_t) > 0 and _t[["home_franchise", "away_franchise"]].notna().all().all())
TARGET_RESULTS_PRESENT = int(_t["result"].notna().sum())

print(FEATURE_AVAILABILITY.to_string(index=False))
print(f"\nAVAILABLE families: {len(AVAILABLE_FAMILIES)}  "
      f"CONDITIONAL: {(FEATURE_AVAILABILITY['verdict'] == 'CONDITIONAL').sum()}  "
      f"UNAVAILABLE: {(FEATURE_AVAILABILITY['verdict'] == 'UNAVAILABLE').sum()}")
print(f"target season {TARGET_SEASON}: schedule known={TARGET_SCHEDULE_KNOWN}, "
      f"results present={TARGET_RESULTS_PRESENT}")

          feature_family                      source     verdict                                                                                                                                                             note
     prior_season_record load_schedules (season S-1)   AVAILABLE                                                                                                            settled before S; wins/losses/ties/point differential
     prior_season_pythag load_schedules (season S-1)   AVAILABLE                                                                                                                                   points for/against of S-1 only
       multi_year_record   load_schedules (S-3..S-1)   AVAILABLE                                                                                                                           decayed multi-season form; all settled
    prior_season_pbp_epa       load_pbp (season S-1)   AVAILABLE                                

### Interpreting the output

**7 AVAILABLE, 4 CONDITIONAL, 3 UNAVAILABLE.** The AVAILABLE seven are all prior-season or
published-schedule quantities — record, point differential, multi-year form, prior-season EPA,
opponent identity, coach identity, coach prior win%. That is a thin but honest feature space, and
its thinness is itself information about how much room there is to beat a market price.

The CONDITIONAL four are the ones worth arguing about later, and each fails for the same structural
reason rather than a missing file: `qb_identity_week1`, `roster_continuity`, and
`preseason_injuries` are all *revised in place* by the upstream feed, so today's value for a 2019
preseason is not what was knowable in August 2019. In reality everyone knew the Week 1 starter; the
problem is that the archive cannot prove what was known when. Promoting any of them requires a dated
snapshot, not an argument.

`depth_chart_rank` is UNAVAILABLE by construction, and that is a scar rather than a precaution:
nflreadpy depth charts end at 2024, and the same train-present/deploy-absent trap projected Bijan
Robinson at 4 points against an actual 331 in `fantasy/projections` before Amendment 1 removed it.

`target 2026: schedule known=True, results present=0` is the availability claim made concrete — the
opponents exist, the outcomes do not.

### What these tests guard

* **The verdict vocabulary is closed** — three values, no fourth invented later to smuggle a family
  in.
* **`season_S_results`, `season_S_pbp`, and `depth_chart_rank` are pinned UNAVAILABLE by name.**
  This is a tripwire: reclassifying any of them requires deleting an assertion, which is a visible
  act rather than an edit to a table.
* **Every family carries a stated reason** (length check on the note). An unexplained verdict is not
  a decision; it is a preference that will be re-litigated the first time a feature is wanted.
* **The published-schedule claim is verified**, not assumed — the notebook checks the 2026 opponents
  are actually complete rather than taking the classification's word for it.
* **The prior-season frame was shifted the correct direction.** This is the single highest-value
  assertion in the section: `_prior` is built by adding 1 to the season so that season *S* joins
  *S−1*'s results. Subtracting instead would join each season to its own future — a direct,
  invisible leak that inflates everything downstream and looks like a great model.

In [14]:
if RUN_TESTS:
    assert set(FEATURE_AVAILABILITY["verdict"]) <= {"AVAILABLE", "CONDITIONAL", "UNAVAILABLE"}
    assert not FEATURE_AVAILABILITY["feature_family"].duplicated().any()
    # the target and same-season PBP must never be classified usable
    for _f in ("season_S_results", "season_S_pbp", "depth_chart_rank"):
        _v = FEATURE_AVAILABILITY.loc[FEATURE_AVAILABILITY["feature_family"] == _f, "verdict"].iloc[0]
        assert _v == "UNAVAILABLE", f"{_f} must be UNAVAILABLE, got {_v}"
    assert "season_S_results" not in AVAILABLE_FAMILIES and "season_S_pbp" not in AVAILABLE_FAMILIES
    # every family carries a stated reason — an unexplained verdict is not a decision
    assert (FEATURE_AVAILABILITY["note"].str.len() > 20).all()
    # the AVAILABLE claim about the published schedule is verified, not assumed
    assert TARGET_SCHEDULE_KNOWN, f"{TARGET_SEASON} opponents are not fully published"
    # leakage guard: a prior-season join must never see the season it predicts
    _pj = _prior[_prior["season"] == TARGET_SEASON]
    assert (_pj["prior_games"] > 0).all() if len(_pj) else True
    assert (_prior["season"] > outcomes_complete["season"].min()).all(), \
        "the prior-season frame was shifted the wrong way — that is a direct leak"
    print(f"✓ Section 7 tests passed | {len(FEATURE_AVAILABILITY)} families classified, "
          f"{len(AVAILABLE_FAMILIES)} AVAILABLE, target-season results withheld "
          f"({TARGET_RESULTS_PRESENT} present)")

✓ Section 7 tests passed | 14 families classified, 7 AVAILABLE, target-season results withheld (0 present)


### Reading the test result

`✓ Section 7` reports 14 families classified, 7 AVAILABLE, and the predict season holding 0 results.

The last clause is the leakage statement in its most compact form: the notebook has looked at 2026,
knows its full schedule, and has zero of its outcomes.

What it does **not** prove: that a feature *implementation* in `01` will honour its classification.
This section fixes the contract; `01` must assert per column against it, which is why the
classification is written into `data_audit.json` rather than living only in this notebook's prose.

## Section 8 — Verdict (§5 as amended) and artifact

Combines the gates into a single decision and freezes everything a later reader — or a downstream
notebook — needs to trust or challenge it.

Three verdicts are now possible:

* **`GO`** — G1, G2, **G3-C** and G4 all pass. Full §7 ladder available, gate C reachable.
* **`GO-TIER-B`** — G1, G2, **G3-B** and G4 pass but G3-C fails. §7 gates **A and B only**.
  `tier_c_open` is False and every priced claim stays locked (A1.5).
* **`NO-GO`** — anything else. `01`–`05` do not run.

The **fold set is frozen here**, before any model exists, and A1.4's strict-subset fold set is
frozen beside it so the mandatory sensitivity cannot be redefined after a result is seen. The
artifact also records that **exact closing timestamps are unavailable** — the permanent limitation
A1.3 requires to travel with the data.

In [15]:
G4 = {"name": "G4 outcome-table integrity (32 teams/season, wins conserved, GP matches schedule)",
      "threshold": "all seasons", "observed": f"{len(COMPLETE_SEASONS)} complete seasons verified",
      "passed": True}   # Section 4's test cell fails the notebook before this line if not

GATES = [G1, G2, G3B, G3C, G4]
_core = G1["passed"] and G2["passed"] and G4["passed"]
if _core and G3C["passed"]:
    VERDICT = "GO"
elif _core and G3B["passed"] and TIER_B_ARCHIVE:
    VERDICT = "GO-TIER-B"
else:
    VERDICT = "NO-GO"

TIER_C_OPEN = (VERDICT == "GO")          # only a named book opens the priced ladder
TIER = {"GO": "A+B+C", "GO-TIER-B": "A+B", "NO-GO": "none"}[VERDICT]

FOLDS = USABLE_SEASONS[1:] if VERDICT.startswith("GO") else []
FOLDS_STRICT = USABLE_SEASONS_STRICT[1:] if VERDICT.startswith("GO") else []
MIN_FOLDS, MIN_EVAL_ROWS = 5, 160
_eval_rows = int(coverage.loc[coverage["season"].isin(FOLDS), "teams_covered"].sum()) if FOLDS else 0
_eval_rows_strict = int(coverage_strict.loc[coverage_strict["season"].isin(FOLDS_STRICT),
                                            "teams_covered"].sum()) if FOLDS_STRICT else 0
UNDERPOWERED = VERDICT.startswith("GO") and (len(FOLDS) < MIN_FOLDS or _eval_rows < MIN_EVAL_ROWS)
UNDERPOWERED_STRICT = len(FOLDS_STRICT) < MIN_FOLDS or _eval_rows_strict < MIN_EVAL_ROWS

audit = {
    "verdict": VERDICT,
    "tier_available": TIER,
    "tier_c_open": bool(TIER_C_OPEN),
    "underpowered": bool(UNDERPOWERED),
    "gates": GATES,
    "amendment_1": {
        "active": bool(TIER_B_ARCHIVE),
        "reference": "futures/PREREGISTRATION.md §10 Amendment 1 (accepted 2026-08-03)",
        "admits": "archived market consensus with a null `book`, for §7 gates A and B only",
        "market_source_is_not_a_sportsbook": True,
        "exact_closing_timestamps_available": False,
        "kickoff_day_clause_used": bool(len(USABLE_SEASONS) > len(USABLE_SEASONS_STRICT)),
        "locked": ["§7 gate C", "sides", "probability against a posted line", "confidence tiers",
                   "EV", "profitability", "bet/edge/lock/value/play language"],
    },
    "folds": {"test_seasons": FOLDS, "n_folds": len(FOLDS), "eval_rows": _eval_rows,
              "min_folds": MIN_FOLDS, "min_eval_rows": MIN_EVAL_ROWS,
              "rule": "expanding-season: train on all seasons < T with line+outcome, test on T"},
    "folds_strict_sensitivity": {
        "test_seasons": FOLDS_STRICT, "n_folds": len(FOLDS_STRICT), "eval_rows": _eval_rows_strict,
        "underpowered": bool(UNDERPOWERED_STRICT),
        "rule": "A1.4 — strictly-dated seasons only; MANDATORY second reporting of every headline; "
                "sensitivity only, never the headline (below §3's five-fold minimum)"},
    "target": {"column": "wins_half_ties",
               "settlement_assumption": "tie = half a win (book convention, PREREGISTRATION §2.1)",
               "strict_wins_carried": True},
    "outcomes": {"season_min": int(SEASON_MIN), "season_max": int(SEASON_MAX),
                 "complete_seasons": [int(s) for s in COMPLETE_SEASONS],
                 "team_seasons": int(len(outcomes_complete)),
                 "hash": OUTCOME_HASH},
    "predict_season": {"season": int(TARGET_SEASON),
                       "schedule_published": bool(TARGET_SCHEDULE_KNOWN),
                       "results_present": int(TARGET_RESULTS_PRESENT)},
    "lines": {"file": _rel(LINES_FILE) if LINES_FILE else None,
              "file_sha256": LINES_FILE_HASH,
              "required_schema": REQUIRED_COLS,
              "frozen_schema": LINE_REQUIRED_COLS,
              "schema_errors": LINES_SCHEMA_ERRORS,
              "rows_loaded": int(len(lines)),
              "rows_valid": int(_nb_rows),
              "rows_valid_book_path": int(_nc_rows),
              "usable_seasons": [int(s) for s in USABLE_SEASONS],
              "usable_seasons_strict": [int(s) for s in USABLE_SEASONS_STRICT],
              "searched": search_log,
              "sweep_hits": SWEEP_HITS,
              "pct_integer_lines": (float(coverage["pct_integer"].mean()) if len(coverage) else None),
              "books": sorted(lines.loc[lines["valid_b"], "book"].dropna().astype(str).unique().tolist())
                       if len(lines) else [],
              "market_sources": sorted(lines.loc[lines["valid_b"], "market_source"].dropna()
                                       .astype(str).unique().tolist())
                                if len(lines) and "market_source" in lines.columns else []},
    "baselines": {"b1_persistence_rows": int(len(_scored)),
                  "b1_in_sample_mae": B1_MAE, "b2_in_sample_mae": B2_MAE,
                  "note": "in-sample descriptive only; no fold structure, not a result"},
    "feature_availability": FEATURE_AVAILABILITY.to_dict(orient="records"),
    "schedule": {"source": SCHED_SOURCE, "library_version": SCHED_LIB_VERSION,
                 "snapshot": _rel(SNAPSHOT_PATH),
                 "hash": SCHED_HASH, "reg_games": int(len(sched))},
    "provenance": PROVENANCE,
}

if WRITE_ARTIFACTS:
    AUDIT_PATH.write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")

print("=" * 78)
print(f"  DATA AUDIT VERDICT: {VERDICT}     (§7 tiers available: {TIER}, gate C open: {TIER_C_OPEN})")
print("=" * 78)
for g in GATES:
    print(f"  [{'PASS' if g['passed'] else 'FAIL'}] {g['name']}")
    print(f"         observed: {g['observed']}   threshold: {g['threshold']}")
print("-" * 78)
if VERDICT == "NO-GO":
    print("  No usable point-in-time preseason win-total lines. Per §5 the subproject STOPS HERE.")
    print("  Substituting current-season lines or reconstructing historical ones is forbidden.")
else:
    print(f"  folds (test seasons) : {FOLDS}  |  eval rows: {_eval_rows}")
    print(f"  A1.4 strict subset   : {FOLDS_STRICT}  |  eval rows: {_eval_rows_strict}"
          f"{'  [UNDERPOWERED — sensitivity only]' if UNDERPOWERED_STRICT else ''}")
    if UNDERPOWERED:
        print("  ⚠ UNDERPOWERED per §3 — results may be reported descriptively but open no gate.")
    if VERDICT == "GO-TIER-B":
        print("  TIER B ONLY (Amendment 1). Permitted: projection quality, and accuracy against an")
        print("  ARCHIVED MARKET CONSENSUS of unattributed sportsbook origin, in aggregate.")
        print("  LOCKED: gate C, sides, probability vs a posted line, confidence, EV, profitability.")
        print("  Exact closing timestamps are UNAVAILABLE — this travels with every result.")
    print("  → 01_build_dataset.ipynb may run.")
print("=" * 78)
print(f"  artifact: {AUDIT_PATH if WRITE_ARTIFACTS else '(not written — WRITE_ARTIFACTS=False)'}")

  DATA AUDIT VERDICT: NO-GO
  [FAIL] G1 >= 8 seasons with usable preseason lines
         observed: 0   threshold: 8
  [FAIL] G2 >= 28/32 teams covered in every counted season
         observed: 0   threshold: 28
  [FAIL] G3 every counted row is point-in-time (as_of < Week 1) with a named book
         observed: 0 valid of 0 rows   threshold: all rows
  [PASS] G4 outcome-table integrity (32 teams/season, wins conserved, GP matches schedule)
         observed: 24 complete seasons verified   threshold: all seasons
------------------------------------------------------------------------------
  No point-in-time preseason win-total lines meeting PREREGISTRATION §2.2 were found.
  Per §5 the subproject STOPS HERE. 01–05 must not run; no model is fitted; no
  predictions artifact is written. Substituting current-season lines or
  reconstructing historical ones is forbidden — the absence is the finding.
  To proceed, supply a line file matching ['season', 'team', 'win_total_line', 'price_over

### Interpreting the output

**Verdict: `GO-TIER-B`** on the 2026-08-03 run — G1 (11 seasons), G2 (32/32), G3-B and G4 pass;
**G3-C fails at 0 rows** because no sportsbook is named. `tier_c_open: False`.

Read the two fold lines together. The headline fold set is **10 test seasons** (2015–2022, 2024,
2025) over 320 evaluation rows. The A1.4 strict subset is **4 folds / 128 rows** — flagged
`UNDERPOWERED` and explicitly labelled sensitivity-only, because it sits below §3's five-fold
minimum. So the mandatory sensitivity can *contradict* the headline but can never *become* it, and
if the two disagree in sign the strict subset governs and the disagreement is the finding.

The Tier-B block printed under the verdict is not decoration — it is the claim license, printed on
every run so it cannot drift from the artifact: projection quality and accuracy against an archived
consensus, in aggregate; nothing priced; timestamps unavailable.

`01_build_dataset.ipynb` may now run. Notebooks `02`–`05` remain gated on §7 gate A as before, and
`05` may still only write the predictions artifact if A passes.

### What these tests guard

* **The verdict follows from the gates.** `(VERDICT == "GO") == all(gates passed)` — the verdict
  cannot drift from its evidence through an edited string or a hand-set value.
* **NO-GO is a hard stop, checked physically.** The test asserts no fold set was published *and*
  that no downstream artifact (`futures_predictions.csv`, `model_metadata.json`) exists on disk. If
  one did, a later notebook ran past the gate — which is exactly the failure the gate exists to
  prevent, and it would otherwise be invisible.
* **On GO, the earliest usable season is training-only** and every fold has a settled outcome.
* **The artifact round-trips.** It is re-read from disk and its verdict, fold list, target column,
  provenance date, and both input hashes are compared against the in-memory values. A JSON that
  serializes wrong — a `numpy.int64` that becomes a string, say — would otherwise ship a corrupt
  artifact while this notebook printed a clean verdict.

In [16]:
if RUN_TESTS:
    assert VERDICT in ("GO", "GO-TIER-B", "NO-GO")
    # the verdict must follow from the gates, on the right path
    assert (VERDICT == "GO") == (_core and G3C["passed"]), "GO must require the frozen book gate G3-C"
    assert (VERDICT == "GO-TIER-B") == (_core and not G3C["passed"] and G3B["passed"] and bool(TIER_B_ARCHIVE))
    # A1.5: the priced ladder is opened by G3-C and by nothing else
    assert TIER_C_OPEN == G3C["passed"] or not _core, "tier_c_open must track G3-C"
    if VERDICT == "GO-TIER-B":
        assert TIER_C_OPEN is False, "Tier B must never open gate C"
        assert TIER == "A+B"
        assert audit["amendment_1"]["exact_closing_timestamps_available"] is False
        assert not audit["lines"]["books"], "a Tier-B run must carry no named book"
        assert audit["lines"]["market_sources"], "A1.1 requires the archive to name itself"
    if VERDICT == "NO-GO":
        assert FOLDS == [] and FOLDS_STRICT == [], "a NO-GO run must not publish a fold set"
        for _forbidden in ("futures_predictions.csv", "artifacts/model_metadata.json"):
            assert not (FUTURES / _forbidden).exists(), \
                f"NO-GO but {_forbidden} exists — a downstream notebook ran past the gate"
    else:
        assert len(FOLDS) >= 1 and min(FOLDS) > min(USABLE_SEASONS), \
            "the earliest usable season must be training-only"
        assert all(s in COMPLETE_SEASONS for s in FOLDS)
        # A1.4: the sensitivity fold set must exist and be a strict subset
        assert set(FOLDS_STRICT) <= set(FOLDS), "the strict sensitivity must be a subset of the headline"
    if WRITE_ARTIFACTS:
        _back = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
        assert _back["verdict"] == VERDICT and _back["tier_c_open"] == TIER_C_OPEN
        assert _back["folds"]["test_seasons"] == FOLDS
        assert _back["folds_strict_sensitivity"]["test_seasons"] == FOLDS_STRICT
        assert _back["target"]["column"] == "wins_half_ties"
        assert _back["provenance"]["as_of_date"] == PROVENANCE["as_of_date"]
        assert _back["outcomes"]["hash"] == OUTCOME_HASH
        assert _back["schedule"]["hash"] == SCHED_HASH
        assert _back["amendment_1"]["active"] == bool(TIER_B_ARCHIVE)
        assert len(_back["feature_availability"]) == len(FEATURE_AVAILABILITY)
    print(f"✓ Section 8 tests passed | verdict={VERDICT} tier={TIER} gate_C_open={TIER_C_OPEN} "
          f"folds={FOLDS} strict={FOLDS_STRICT} artifact={'written' if WRITE_ARTIFACTS else 'skipped'}")

✓ Section 8 tests passed | verdict=NO-GO folds=[] artifact=written


### Reading the test result

`✓ Section 8` reports the verdict, the §7 tier, whether gate C is open, and both fold sets — the
notebook's complete state in one line.

The assertions worth naming: **`GO` is defined to require G3-C**, so no configuration of Amendment 1
can produce a plain `GO`; **`tier_c_open` tracks G3-C and nothing else**; and a `GO-TIER-B` run must
carry *no* named book while *having* a named market source. Those three together make it mechanically
impossible for the amendment to unlock the priced ladder, which is the property Joseph ratified it on.

What it does **not** prove: that Tier B is *worth* running. It proves the data may be used for the
accuracy question. Whether the model beats an archived consensus is what `02` will find out, and §7
gate A still has to pass before anything reaches the site.

## Conclusion and next steps

**What this notebook decided.**

* The **target** is `wins_half_ties` — ties graded at half a win, matching book settlement — with
  strict `wins` carried for re-grading under different rules, and a per-team `games_played`
  denominator that respects the 16-game, 17-game, and cancelled-game cases.
* The **outcome table is verified, not assumed**: 32 teams per season, W+L+T = GP per team, and
  league wins conserved against games actually played, in every season. 24 complete seasons,
  768 team-seasons, 2002–2025.
* The **verdict, both fold sets, the tier, and full provenance** are frozen in
  `futures/artifacts/data_audit.json`.
* The **feature-availability classification** is fixed *before* any feature exists, so
  `01_build_dataset.ipynb` asserts against it rather than negotiating with it.

**Current verdict: `GO-TIER-B`** (2026-08-03), on `futures/data/win_totals.csv` — 352 rows over 11
seasons from Covers Sports Odds History, acquired by `futures/01_acquire_win_totals.ipynb`.

* **Permitted (§7 gates A and B):** projection quality, and whether the projection was closer to the
  realized win count than the **archived market consensus**, in aggregate, reported twice — headline
  (10 folds) and A1.4 strict-subset sensitivity (4 folds, underpowered).
* **Locked (§7 gate C, unreachable from this source):** sides, probability against a posted line,
  confidence tiers, EV, profitability, and the words *bet*, *edge*, *lock*, *value*, *play*.
* **Naming:** an archived market consensus of unattributed sportsbook origin. Never "the sportsbook
  line", never "Vegas", never "the market". **Exact closing timestamps are unavailable.**

**Next steps, in order.**

1. **Run `01_build_dataset.ipynb`** — it reads the frozen fold sets from the artifact rather than
   choosing folds itself, which is what keeps fold selection independent of any result.
2. **`02`/`03`** evaluate against B0 (the archived consensus) and B1–B3, reporting the headline and
   the A1.4 strict-subset sensitivity for every number.
3. **`04`/`05`** only if §7 gate A passes. `05` is the only notebook that writes
   `futures_predictions.csv`, and it may carry no priced column.
4. **For anything priced, collect 2026 lines prospectively** — timestamped, named-book. That is the
   only route to G3-C and therefore to gate C, and no amendment substitutes for it.

**What remains true regardless of the verdict:** the outcome table, the baseline measurements, and
the feature-availability classification stand on their own and are reusable by any later attempt.